In [1]:
import numpy as np
import sympy as sp
from sympy import sqrt, zeros, S,  Matrix, binomial
import matplotlib.pyplot as plt
from sympy.physics.quantum import TensorProduct
from sympy.physics.quantum.trace import Tr
from sympy import lambdify, simplify
from math import comb, isnan
%matplotlib inline

In [2]:
# Identity operators for ion and photon Hilbert spaces
id_ion = sp.eye(2)
id_photon = sp.eye(5)

# Basis vectors for the ion
ket_0_ion = sp.Matrix([1, 0])
ket_1_ion = sp.Matrix([0, 1])

bra_0_ion = ket_0_ion.T
bra_1_ion = ket_1_ion.T

# Basis vectors for the photon (Assume there is at most four photon at each optical mode)
ket_0_photon = sp.Matrix([1, 0, 0, 0, 0])
ket_1_photon = sp.Matrix([0, 1, 0, 0, 0])
ket_2_photon = sp.Matrix([0, 0, 1, 0, 0])
ket_3_photon = sp.Matrix([0, 0, 0, 1, 0])
ket_4_photon = sp.Matrix([0, 0, 0, 0, 1])

bra_0_photon = ket_0_photon.T
bra_1_photon = ket_1_photon.T
bra_2_photon = ket_2_photon.T
bra_3_photon = ket_3_photon.T
bra_4_photon = ket_4_photon.T

# Define Bell states for the ions
phi_plus = (1/sqrt(2)) * (TensorProduct(ket_0_ion, ket_0_ion) + TensorProduct(ket_1_ion, ket_1_ion))
psi_plus = (1/sqrt(2)) * (TensorProduct(ket_0_ion, ket_1_ion) + TensorProduct(ket_1_ion, ket_0_ion))
psi_minus = (1/sqrt(2)) * (TensorProduct(ket_0_ion, ket_1_ion) - TensorProduct(ket_1_ion, ket_0_ion))
phi_minus = (1/sqrt(2)) * (TensorProduct(ket_0_ion, ket_0_ion) - TensorProduct(ket_1_ion, ket_1_ion))

In [3]:
def normalize(dm):
    '''
    Return the normalized version of density matrix.
    '''
    return dm / Tr(dm)

def get_heralded_dm_spdc(dm):
    '''
    Return the "un-normalized" density matrix after the center swapping station for SPDC clicks. Assume photon-number-resolving detectors.
    
    The state is |QM1, Fiber1, QM2, Fiber2>

    We consider only one possible detection pattern. Other detection patterns are locally equivalent to this one up to local Z rotation.
    '''

    # Output mode written in the basis of input mode.
    o_10 = (1/sqrt(2)) * (TensorProduct(id_photon, ket_0_photon, id_photon, ket_1_photon) + TensorProduct(id_photon, ket_1_photon, id_photon, ket_0_photon))
    rho_10 = o_10.T @ dm @ o_10

    return rho_10


def get_heralded_dm_load_1(dm):
    '''
    Return the "un-normalized" density matrix after the loading to first ion-trap. Assume number-resolving detectors.
    
    The state is |QM1, QM2, ion1, photon1>
    '''

    # Output mode written in the basis of input mode.
    
    o_10 = (1/sqrt(2)) * (1 * TensorProduct(ket_1_photon, id_photon, id_ion, ket_0_photon) + 1 * TensorProduct(ket_0_photon, id_photon, id_ion, ket_1_photon))


    # The heralded QM state with different photon numbers.
    
    rho_10 = o_10.T @ dm @ o_10


    return rho_10

def get_heralded_dm_load_2(dm):
    '''
    Return the "un-normalized" density matrix after the loading to second ion-trap. Assume number-resolving detectors.
    
    The state is |QM2, ion1, ion2, photon2>
    '''

    # Output mode written in the basis of input mode.
    
    o_10 = (1/sqrt(2)) * (1 * TensorProduct(ket_1_photon, id_ion, id_ion, ket_0_photon) + 1 * TensorProduct(ket_0_photon, id_ion, id_ion, ket_1_photon))


    # The heralded QM state with different photon numbers.
    
    rho_10 = o_10.T @ dm @ o_10
    
    return rho_10 

def amplitude_damping_kraus_fiber(d):
    '''
    Generate the Kraus operators for the amplitude damping channel 

    Input:
    d: dimension of the bosonic state. (Maximum number of photons = d-1)
    '''
    
    kraus_ops = []
    for k in range(d):
        K = Matrix.zeros(d, d)
        for n in range(k, d):
            amp = sqrt(binomial(n, k)) * ((1 - p)**((n - k)/2)) * (p**(k/2))
            K[n - k, n] = amp
        kraus_ops.append(K)
    
    return kraus_ops

def amplitude_damping_kraus_qm(d):
    '''
    Generate the Kraus operators for the amplitude damping channel 

    Input:
    d: dimension of the bosonic state. (Maximum number of photons = d-1)
    '''
    
    kraus_ops = []
    for k in range(d):
        K = Matrix.zeros(d, d)
        for n in range(k, d):
            amp = sqrt(binomial(n, k)) * ((1 - p_m)**((n - k)/2)) * (p_m**(k/2))
            K[n - k, n] = amp
        kraus_ops.append(K)
    
    return kraus_ops

def amplitude_damping_kraus_ion_emission(d):
    '''
    Generate the Kraus operators for the amplitude damping channel 

    Input:
    d: dimension of the bosonic state. (Maximum number of photons = d-1)
    '''
    
    kraus_ops = []
    for k in range(d):
        K = Matrix.zeros(d, d)
        for n in range(k, d):
            amp = sqrt(binomial(n, k)) * ((1 - p_i)**((n - k)/2)) * (p_i**(k/2))
            K[n - k, n] = amp
        kraus_ops.append(K)
    
    return kraus_ops


def apply_ED_protocol(dm):

    input_dm = TensorProduct(dm, dm)

    x_gate = Matrix([[0,1], [1,0]])

    # |a, b, a, b>
    CNOT_alice = TensorProduct(ket_0_ion @ bra_0_ion, id_ion, id_ion, id_ion) + TensorProduct(ket_1_ion @ bra_1_ion, id_ion, x_gate, id_ion)
    CNOT_bob = TensorProduct(id_ion, ket_0_ion @ bra_0_ion, id_ion, id_ion) + TensorProduct(id_ion, ket_1_ion @ bra_1_ion, id_ion, x_gate)

    dm_cnot = CNOT_bob @ (CNOT_alice @ input_dm @ CNOT_alice.T) @ CNOT_bob.T
    post_select_dm = TensorProduct(id_ion, id_ion, bra_1_ion, bra_1_ion) * dm_cnot * TensorProduct(id_ion, id_ion, ket_1_ion, ket_1_ion)
    prob = Tr(post_select_dm)
    return normalize(post_select_dm), prob

def get_p_vec(dm):
    dm_np = np.array(dm)
    pI = (phi_plus.T @ dm_np @ phi_plus)[0]
    pX = (psi_plus.T @ dm_np @ psi_plus)[0]
    pY = (psi_minus.T @ dm_np @ psi_minus)[0]
    pZ = (phi_minus.T @ dm_np @ phi_minus)[0]

    # Assume local rotation back to pI (change Pauli frame)
    return np.array([pX, pI, pZ, pY])

def get_SPDC_coeffs(lam):

    N0 = np.sqrt(1-lam**2)

    c0 = N0 * (lam ** 0)
    c1 = N0 * (lam ** 1)
    c2 = N0 * (lam ** 2)
    
    length = np.sqrt(c0**2 + c1**2 + c2**2)

    return c0/length, c1/length, c2/length


def fidelity(dm):
    return get_p_vec(dm)[0]

In [4]:
# Components for the SPDC state, up to two-photon emission.
p0 = sp.symbols('p_0')
p1 = sp.symbols('p_1')
p2 = sp.symbols('p_2')

# Brightness of the ion.
q = sp.symbols('q')

# The total loss for one-arm (remote)
p = sp.symbols('p')

# The memory loss during readout
p_m = sp.symbols('p_m')

# The photon loss for ion-trap emission.
p_i = sp.symbols('p_i')

# Two-mode squeezed vacuum state for SPDC.(Truncated to four photons.)
sp_joint_spdc_ket =  p0**2 * TensorProduct(ket_0_photon, ket_0_photon, ket_0_photon, ket_0_photon) + (p0 * p1) * TensorProduct(ket_0_photon, ket_0_photon, ket_1_photon, ket_1_photon) + (p0 * p1) * TensorProduct(ket_1_photon, ket_1_photon, ket_0_photon, ket_0_photon) + (p0*p2) * TensorProduct(ket_0_photon, ket_0_photon, ket_2_photon, ket_2_photon) + (p0*p2) * TensorProduct( ket_2_photon, ket_2_photon, ket_0_photon, ket_0_photon) + p1**2 * TensorProduct(ket_1_photon, ket_1_photon, ket_1_photon, ket_1_photon)

# Joint SPDC state for both side's SPDC.
sp_joint_dm = normalize(sp_joint_spdc_ket @ sp_joint_spdc_ket.T)

# Define Kraus operators for the remote fiber loss (max. number of photons = 4)
K_ops_fiber = amplitude_damping_kraus_fiber(5)
K_0_fiber = K_ops_fiber[0]
K_1_fiber = K_ops_fiber[1]
K_2_fiber = K_ops_fiber[2]
K_3_fiber = K_ops_fiber[3]
K_4_fiber = K_ops_fiber[4]

# Define Kraus operators for the quantum memory readout loss (max. number of photons = 4) 
K_ops_amp_qm = amplitude_damping_kraus_qm(5)
K_0_amp_qm = K_ops_amp_qm[0]
K_1_amp_qm = K_ops_amp_qm[1]
K_2_amp_qm = K_ops_amp_qm[2]
K_3_amp_qm = K_ops_amp_qm[3]
K_4_amp_qm = K_ops_amp_qm[4]

# Define Kraus operators for the ion emission loss (max. number of photons = 4)
K_ops_amp_ion_emission = amplitude_damping_kraus_ion_emission(5)
K_0_amp_ion_emission = K_ops_amp_ion_emission[0]
K_1_amp_ion_emission = K_ops_amp_ion_emission[1]
K_2_amp_ion_emission = K_ops_amp_ion_emission[2]
K_3_amp_ion_emission = K_ops_amp_ion_emission[3]
K_4_amp_ion_emission = K_ops_amp_ion_emission[4]

# Damp the first fiber photon (State : |QM1, Fiber1, QM2, Fiber2>
sp_joint_dm = TensorProduct(id_photon, K_0_fiber, id_photon, id_photon) @ sp_joint_dm @ TensorProduct(id_photon, K_0_fiber.T, id_photon, id_photon) + TensorProduct(id_photon, K_1_fiber, id_photon, id_photon) @ sp_joint_dm @ TensorProduct(id_photon, K_1_fiber.T, id_photon, id_photon) + TensorProduct(id_photon, K_2_fiber, id_photon, id_photon) @ sp_joint_dm @ TensorProduct(id_photon, K_2_fiber.T, id_photon, id_photon) + TensorProduct(id_photon, K_3_fiber, id_photon, id_photon) @ sp_joint_dm @ TensorProduct(id_photon, K_3_fiber.T, id_photon, id_photon) + TensorProduct(id_photon, K_4_fiber, id_photon, id_photon) @ sp_joint_dm @ TensorProduct(id_photon, K_4_fiber.T, id_photon, id_photon)

# Damp the second fiber photon (State: |QM1, Fiber1, QM2, Fiber2>
sp_joint_dm = TensorProduct(id_photon, id_photon, id_photon, K_0_fiber) @ sp_joint_dm @ TensorProduct(id_photon, id_photon, id_photon, K_0_fiber.T) + TensorProduct(id_photon, id_photon, id_photon, K_1_fiber) @ sp_joint_dm @ TensorProduct(id_photon, id_photon, id_photon, K_1_fiber.T) + TensorProduct(id_photon, id_photon, id_photon, K_2_fiber) @ sp_joint_dm @ TensorProduct(id_photon, id_photon, id_photon, K_2_fiber.T) + TensorProduct(id_photon, id_photon, id_photon, K_3_fiber) @ sp_joint_dm @ TensorProduct(id_photon, id_photon, id_photon, K_3_fiber.T) + TensorProduct(id_photon, id_photon, id_photon, K_4_fiber) @ sp_joint_dm @ TensorProduct(id_photon, id_photon, id_photon, K_4_fiber.T)

# QM-QM entangled state after central swapping. (NOT NORMALIZED!)
sp_qm_dm = get_heralded_dm_spdc(sp_joint_dm)

# Probability of success in the QM swap stage (There is two successful pattern)
sp_prob_click = 2 * Tr(sp_qm_dm)

# Density matrix for the QM-QM state.
sp_qm_dm = normalize(sp_qm_dm)

# Apply readout loss (QM1) (State: |QM1, QM2>)
sp_qm_dm = TensorProduct(K_0_amp_qm, id_photon) @ sp_qm_dm @ TensorProduct(K_0_amp_qm.T, id_photon) + TensorProduct(K_1_amp_qm, id_photon) @ sp_qm_dm @ TensorProduct(K_1_amp_qm.T, id_photon) + TensorProduct(K_2_amp_qm, id_photon) @ sp_qm_dm @ TensorProduct(K_2_amp_qm.T, id_photon) + TensorProduct(K_3_amp_qm, id_photon) @ sp_qm_dm @ TensorProduct(K_3_amp_qm.T, id_photon) + TensorProduct(K_4_amp_qm, id_photon) @ sp_qm_dm @ TensorProduct(K_4_amp_qm.T, id_photon)

# Apply readout loss (QM2) (State: |QM1, QM2>)
sp_qm_dm = TensorProduct(id_photon, K_0_amp_qm) @ sp_qm_dm @ TensorProduct(id_photon, K_0_amp_qm.T) + TensorProduct(id_photon, K_1_amp_qm) @ sp_qm_dm @ TensorProduct(id_photon, K_1_amp_qm.T) + TensorProduct(id_photon, K_2_amp_qm) @ sp_qm_dm @ TensorProduct(id_photon, K_2_amp_qm.T) + TensorProduct(id_photon, K_3_amp_qm) @ sp_qm_dm @ TensorProduct(id_photon, K_3_amp_qm.T) + TensorProduct(id_photon, K_4_amp_qm) @ sp_qm_dm @ TensorProduct(id_photon, K_4_amp_qm.T)

# Ion-photon entangled state from the ion-trap. Assume |0_ion> is the bright state)
sp_ion_photon_ket = sqrt(q) * TensorProduct(ket_0_ion, ket_1_photon) + sqrt(1-q) * TensorProduct(ket_1_ion, ket_0_photon)

# Density matrix for the ion-photon entangled state.
sp_ion_photon_dm = sp_ion_photon_ket @ sp_ion_photon_ket.T

# Apply emission loss 
sp_ion_photon_dm = TensorProduct(id_ion, K_0_amp_ion_emission) @ sp_ion_photon_dm @ TensorProduct(id_ion, K_0_amp_ion_emission.T) + TensorProduct(id_ion, K_1_amp_ion_emission) @ sp_ion_photon_dm @ TensorProduct(id_ion, K_1_amp_ion_emission.T) + TensorProduct(id_ion, K_2_amp_ion_emission) @ sp_ion_photon_dm @ TensorProduct(id_ion, K_2_amp_ion_emission.T) + TensorProduct(id_ion, K_3_amp_ion_emission) @ sp_ion_photon_dm @ TensorProduct(id_ion, K_3_amp_ion_emission.T) + TensorProduct(id_ion, K_4_amp_ion_emission) @ sp_ion_photon_dm @ TensorProduct(id_ion, K_4_amp_ion_emission.T)

# Density matrix for the state |QM1, QM2, ion1, photon1>
sp_joint_dm = TensorProduct(sp_qm_dm, sp_ion_photon_dm)

# Density matrix after first loading. The state is |QM2, ion1> (NOT NORMALIZED!)
sp_joint_dm = get_heralded_dm_load_1(sp_joint_dm)

# Density matrix with another ion-photon entangled state. The state is |QM2, ion1, ion2, photon2>
sp_joint_dm = TensorProduct(sp_joint_dm, sp_ion_photon_dm)

# Density matrix after second loading. The state is |ion1, ion2> (NOT NORMALIZED)
sp_joint_dm = get_heralded_dm_load_2(sp_joint_dm)

# Final ion-ion dm (NOT NORMALIZED!)
sp_ion_ion_dm = sp_joint_dm

# Click probability (Two click patterms on each side, total four)
sp_prob_load = 4 * Tr(sp_ion_ion_dm)

# Final ion-ion dm (NORMALIZED.)
sp_ion_ion_dm_normalized = normalize(sp_ion_ion_dm)

sp_state_ED, sp_prob_ED = apply_ED_protocol(sp_ion_ion_dm_normalized)


In [5]:
def extend_state(ion_ion_dm):
    '''
    Extend the ion-ion entangled dm with |00> photon vacuum state.
    The output state is |ion1, ion2, photon1, photon2>
    '''
    dm_photon1 = ket_0_photon @ bra_0_photon
    dm_photon2 = ket_0_photon @ bra_0_photon
    return TensorProduct(ion_ion_dm, dm_photon1, dm_photon2)

def reemit(ion_dm):
    '''
    Create the re-emitted state from an already established elementary link.
    The input and output state is |ion1, ion2, photon1, photon2>
    '''
    # Extend the ion-ion dm
    extended_ion_dm = extend_state(ion_dm)

    # Flip the ions
    x_gate_ion = ket_0_ion @ bra_1_ion + ket_1_ion @ bra_0_ion
    extended_ion_dm = TensorProduct(x_gate_ion, x_gate_ion, id_photon, id_photon) @ extended_ion_dm @ TensorProduct(x_gate_ion, x_gate_ion, id_photon, id_photon).T

    # Define X-gate (model the emission, acting on photonic degree of freedom)
    x_gate_photon = ket_0_photon @ bra_1_photon + ket_1_photon @ bra_0_photon

    # Reemission as C0NOT gate
    emit_op1 = TensorProduct(ket_0_ion @ bra_0_ion, id_ion, x_gate_photon, id_photon) + TensorProduct(ket_1_ion @ bra_1_ion, id_ion, id_photon, id_photon)
    emit_op2 = TensorProduct(id_ion, ket_0_ion @ bra_0_ion, id_photon, x_gate_photon) + TensorProduct(id_ion, ket_1_ion @ bra_1_ion, id_photon, id_photon)

    emit_joint = emit_op2 @ (emit_op1 @ extended_ion_dm @ emit_op1.T) @ emit_op2.T

    emit_joint = TensorProduct(id_ion, id_ion, K_0_amp_ion_emission, id_photon) @ emit_joint @ TensorProduct(id_ion, id_ion, K_0_amp_ion_emission.T, id_photon) + TensorProduct(id_ion, id_ion, K_1_amp_ion_emission, id_photon) @ emit_joint @ TensorProduct(id_ion, id_ion, K_1_amp_ion_emission.T, id_photon) + TensorProduct(id_ion, id_ion, K_2_amp_ion_emission, id_photon) @ emit_joint @ TensorProduct(id_ion, id_ion, K_2_amp_ion_emission.T, id_photon) + TensorProduct(id_ion, id_ion, K_3_amp_ion_emission, id_photon) @ emit_joint @ TensorProduct(id_ion, id_ion, K_3_amp_ion_emission.T, id_photon) + TensorProduct(id_ion, id_ion, K_4_amp_ion_emission, id_photon) @ emit_joint @ TensorProduct(id_ion, id_ion, K_4_amp_ion_emission.T, id_photon)

    emit_joint = TensorProduct(id_ion, id_ion, id_photon, K_0_amp_ion_emission) @ emit_joint @ TensorProduct(id_ion, id_ion, id_photon, K_0_amp_ion_emission.T) +  TensorProduct(id_ion, id_ion, id_photon, K_1_amp_ion_emission) @ emit_joint @ TensorProduct(id_ion, id_ion, id_photon, K_1_amp_ion_emission.T) +  TensorProduct(id_ion, id_ion, id_photon, K_2_amp_ion_emission) @ emit_joint @ TensorProduct(id_ion, id_ion, id_photon, K_2_amp_ion_emission.T) +  TensorProduct(id_ion, id_ion, id_photon, K_3_amp_ion_emission) @ emit_joint @ TensorProduct(id_ion, id_ion, id_photon, K_3_amp_ion_emission.T) +  TensorProduct(id_ion, id_ion, id_photon, K_4_amp_ion_emission) @ emit_joint @ TensorProduct(id_ion, id_ion, id_photon, K_4_amp_ion_emission.T)

    return emit_joint

def reload(ion_dm, qm_dm):
    '''
    Reload the ion with a new QM entangled pair.
    '''

    # |ion1, ion2, photon1, photon2, QM1, QM2>
    joint_dm = TensorProduct(ion_dm, qm_dm)
    o_10_I = (1/sqrt(2)) * ((1) * TensorProduct(id_ion, id_ion, ket_1_photon, id_photon, ket_0_photon, id_photon) + (1) * TensorProduct(id_ion, id_ion, ket_0_photon, id_photon, ket_1_photon, id_photon))

    rho_10_I = o_10_I.T @ joint_dm @ o_10_I

    joint_dm = rho_10_I

    # |ion1, ion2, photon2, QM2>
    o_10_II = (1/sqrt(2)) * ((1) * TensorProduct(id_ion, id_ion, ket_1_photon, ket_0_photon) + (1) * TensorProduct(id_ion, id_ion, ket_0_photon, ket_1_photon))

    rho_10_II = o_10_II.T @ joint_dm @ o_10_II

    return normalize(rho_10_II)

In [6]:
fid_lst_pnr_reemission = []

q_val = 0.1
p_val = 0.5
p0_val = 0.990
p1_val = 0.099
p2_val = 0.009

local_loss_lst = np.linspace(0, 0.9, 100)

for local_loss in local_loss_lst:
    
    ion_ion_dm_val = sp_ion_ion_dm_normalized.subs({q: q_val, p: p_val, p0: p0_val, p1: p1_val, p2: p2_val, p_m: local_loss, p_i: local_loss})
    qm_dm_val = sp_qm_dm.subs({q: q_val, p: p_val, p0: p0_val, p1: p1_val, p2: p2_val, p_m: local_loss, p_i: local_loss})
    ion_reemit_dm_val = reemit(ion_ion_dm_val)
    reloaded_dm = reload(ion_reemit_dm_val, qm_dm_val).subs({q: q_val, p: p_val, p0: p0_val, p1: p1_val, p2: p2_val, p_m: local_loss, p_i: local_loss})
    fid_lst_pnr_reemission.append(get_p_vec(reloaded_dm)[0])

In [7]:
np.save("fid_lst_pnr_reemission", np.array(fid_lst_pnr_reemission))